# Evaluation for BPI Challenge 2013 (closed problems) models

Inside the folder `<project_root>/runs/bpic2013_closed_problems` we have a list of folders named as `<percentage>%`, where `<percentage>` is the percentage of the dataset used for training the model.
In each of these folders we have a folder named as the best model found during the training phase based on the accuracy value.
Inside each of these folders we have the following files:
- `constraints_satisfaction_rate.csv`: a CSV file containing the constraints satisfaction rate for each of the test traces.
- `constraints_satisfactions.csv`: a CSV file containing the constraints satisfaction for each of the test traces.
- `predicted_traces.txt`: a TXT file containing the traces generated by the model for each of the test traces.
- `predictions.csv`: a CSV file containing the predictions step by step for each of the test traces.
- `results.json`: a JSON file containing the results of the evaluation of the model on the test set.

In [1]:
DATASET_NAME = "bpic2013_closed_problems"

In [2]:
from collections import namedtuple
import pathlib

project_root = pathlib.Path("../../..").parent.resolve()

Info = namedtuple("Info", ["model_args", "model_path", "results_path"])

models_path: dict[int, list[Info]] = {}

for dataset_percentage in range(20, 101, 20):
    checkpoints = [
        path
        for path in (project_root / "runs" / DATASET_NAME).rglob(
            f"{dataset_percentage}%/**/*.best_val_acc.pth"
        )
    ]
    results = [
        pathlib.Path(str(checkpoint).removesuffix(".pth"))
        / "step_by_step"
        / "results.json"
        for checkpoint in checkpoints
    ]
    args = [checkpoint.parent / "args.json" for checkpoint in checkpoints]
    models_path[dataset_percentage] = [
        Info(model_args=args, model_path=checkpoint, results_path=result)
        for args, checkpoint, result in zip(args, checkpoints, results)
    ]

## Comparison

In [3]:
import json
import pandas as pd

pd.set_option("max_colwidth", 400)

dataframes = {}

for percentage in models_path:
    dataframes[percentage] = pd.DataFrame(
        columns=[
            "lr",
            "dropout",
            "loss",
            "acc",
            "dld",
            "norm_dld",
            "constraints",
            "constraints_multiplier",
        ]
    )
    for info in models_path[percentage]:
        with open(info.model_args) as f:
            args = json.load(f)
        try:
            with open(info.results_path) as f:
                results = json.load(f)
                dataframes[percentage].loc[info.model_path.parent.name] = [
                    args["learning_rate"],
                    args["model"]["dropout"],
                    results["loss"],
                    results["acc"],
                    results["dld"],
                    results["norm_dld"],
                    args.get("constraints", None),
                    args.get("constraints_multiplier", None),
                ]
        except FileNotFoundError:
            print(f"Missing results for {info.model_path}")

/tmp/ipykernel_3886096/1466535417.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dataframes[percentage].loc[info.model_path.parent.name] = [
/tmp/ipykernel_3886096/1466535417.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dataframes[percentage].loc[info.model_path.parent.name] = [
/tmp/ipykernel_3886096/1466535417.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA c

In [4]:
for percentage in range(20, 101, 20):
    print("=" * 10 + f" {percentage}% " + "=" * 10)
    display(dataframes[percentage].sort_values("dld", ascending=True))
    print()

========== 20% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.1828.constraints,0.0001,0.2,0.324405,0.774436,2.000000,0.284480,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.001
20250319.1830.constraints,0.0001,0.2,0.324181,0.774436,2.000000,0.284480,"[Succession[Accepted, Completed]*0.99]",0.001
20250319.1833.constraints,0.0001,0.2,0.324103,0.774436,2.000000,0.284480,[Exactly1[Completed]*0.95],0.001
20250319.1834.constraints,0.0001,0.2,0.324218,0.774436,2.000000,0.284480,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.001
20250319.1839.constraints,0.0001,0.2,0.325866,0.774436,2.000000,0.284480,"[Succession[Accepted, Completed]*0.99]",0.010
20250319.1843.constraints,0.0001,0.2,0.326242,0.774436,2.000000,0.284480,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.010
20250319.1836.constraints,0.0001,0.2,0.328480,0.766917,2.066667,0.297813,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.010
20250319.1842.constraints,0.0001,0.2,0.324616,0.751880,2.133333,0.299295,[Exactly1[Completed]*0.95],0.010
20250319.2337.no_constraint,0.0001,0.2,0.321942,0.759399,2.133333,0.299295,[],NaN
20250319.1848.constraints,0.0001,0.2,0.399101,0.751880,2.200000,0.312804,"[Succession[Accepted, Completed]*0.99]",0.100



========== 40% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.2338.no_constraint,0.0001,0.2,0.236306,0.766129,2.590909,0.228945,[],NaN
20250319.1921.constraints,0.0001,0.2,0.239647,0.758065,2.636364,0.241135,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.010
20250319.1927.constraints,0.0001,0.2,0.237841,0.754032,2.727273,0.259500,"[Succession[Accepted, Completed]*0.99]",0.010
20250319.1933.constraints,0.0001,0.2,0.237722,0.754032,2.727273,0.259500,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.010
20250319.1950.constraints,0.0001,0.2,0.310965,0.758065,2.727273,0.244412,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.100
20250319.1944.constraints,0.0001,0.2,0.307629,0.758065,2.727273,0.234311,"[Succession[Accepted, Completed]*0.99]",0.100
20250319.1932.constraints,0.0001,0.2,0.236679,0.754032,2.727273,0.253187,[Exactly1[Completed]*0.95],0.010
20250319.1915.constraints,0.0001,0.2,0.236989,0.750000,2.772727,0.264551,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.001
20250319.1903.constraints,0.0001,0.2,0.237185,0.750000,2.772727,0.264551,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.001
20250319.1908.constraints,0.0001,0.2,0.237012,0.750000,2.772727,0.264551,"[Succession[Accepted, Completed]*0.99]",0.001



========== 60% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.2011.constraints,0.0001,0.2,0.404682,0.712589,2.510638,0.311614,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.001
20250319.2019.constraints,0.0001,0.2,0.404472,0.710214,2.531915,0.313741,"[Succession[Accepted, Completed]*0.99]",0.001
20250319.2025.constraints,0.0001,0.2,0.404470,0.710214,2.531915,0.313741,[Exactly1[Completed]*0.95],0.001
20250319.2027.constraints,0.0001,0.2,0.404515,0.710214,2.531915,0.313741,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.001
20250319.2042.constraints,0.0001,0.2,0.404436,0.707838,2.553191,0.335018,"[Succession[Accepted, Completed]*0.99]",0.010
20250319.2048.constraints,0.0001,0.2,0.405004,0.707838,2.574468,0.337146,[Exactly1[Completed]*0.95],0.010
20250319.2050.constraints,0.0001,0.2,0.404616,0.705463,2.595745,0.340185,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.010
20250319.2102.constraints,0.0001,0.2,0.479440,0.707838,2.595745,0.339926,"[Succession[Accepted, Completed]*0.99]",0.100
20250319.2034.constraints,0.0001,0.2,0.404742,0.700713,2.617021,0.329254,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.010
20250319.2107.constraints,0.0001,0.2,0.411211,0.703088,2.617021,0.361462,[Exactly1[Completed]*0.95],0.100



========== 80% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.2241.constraints,0.0001,0.2,0.415671,0.712381,2.354839,0.339494,[Exactly1[Completed]*0.95],0.100
20250319.2215.constraints,0.0001,0.2,0.407585,0.710476,2.370968,0.355623,[Exactly1[Completed]*0.95],0.010
20250319.2340.no_constraint,0.0001,0.2,0.413216,0.712381,2.370968,0.355623,[],NaN
20250319.2152.constraints,0.0001,0.2,0.407117,0.708571,2.387097,0.361000,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.001
20250319.2217.constraints,0.0001,0.2,0.402498,0.708571,2.387097,0.361000,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.010
20250319.2150.constraints,0.0001,0.2,0.407139,0.708571,2.387097,0.361000,[Exactly1[Completed]*0.95],0.001
20250319.2143.constraints,0.0001,0.2,0.406813,0.708571,2.387097,0.361000,"[Succession[Accepted, Completed]*0.99]",0.001
20250319.2208.constraints,0.0001,0.2,0.403623,0.704762,2.451613,0.363978,"[Succession[Accepted, Completed]*0.99]",0.010
20250319.2135.constraints,0.0001,0.2,0.407326,0.704762,2.451613,0.363978,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.001
20250319.2200.constraints,0.0001,0.2,0.406349,0.702857,2.467742,0.367204,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.010



========== 100% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.1129.constraints,0.0001,0.2,0.375325,0.715047,2.890411,0.298553,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.010
20250319.1531.constraints,0.0001,0.2,0.440902,0.711052,2.890411,0.325380,"[Succession[Accepted, Completed]*0.99]",0.100
20250319.1545.constraints,0.0001,0.2,0.459523,0.713715,2.890411,0.312823,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.100
20250319.1519.constraints,0.0001,0.2,0.375021,0.712384,2.917808,0.303130,"[Succession[Accepted, Completed]*0.99, Exactly1[Completed]*0.95]",0.010
20250319.1504.constraints,0.0001,0.2,0.376767,0.712384,2.917808,0.301988,"[Succession[Accepted, Completed]*0.99]",0.010
20250319.1226.constraints,0.0001,0.2,0.378396,0.709720,2.931507,0.302902,"[Succession[Accepted, Completed]*0.99, Chain Precedence[Accepted, Completed]*0.99, Init[Accepted]*0.96, Exactly1[Completed]*0.95, Absence2[Queued]*0.88]",0.001
20250319.1439.constraints,0.0001,0.2,0.378448,0.708389,2.945205,0.309751,"[Succession[Accepted, Completed]*0.99]",0.001
20250319.1124.no_constraint,0.0001,0.2,0.379214,0.708389,2.945205,0.310892,[],NaN
20250319.1516.constraints,0.0001,0.2,0.378553,0.708389,2.945205,0.310892,[Exactly1[Completed]*0.95],0.010
20250319.1542.constraints,0.0001,0.2,0.382697,0.707057,2.945205,0.303167,[Exactly1[Completed]*0.95],0.100
